In [8]:
!pip install -q openai-whisper
!pip install -q language-tool-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.7 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00


In [9]:
import os, gc, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import whisper
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

In [10]:
TRAIN_CSV   = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/csvs/train.csv"
TEST_CSV    = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/csvs/test.csv"
TRAIN_AUDIO = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/train"
TEST_AUDIO  = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/test"

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print(train_df.head())


Train shape: (409, 2)
Test shape : (197, 1)
    filename  label
0  audio_173    3.0
1  audio_138    3.0
2  audio_127    2.0
3   audio_95    2.0
4   audio_73    3.5


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [21]:
!pip install faster-whisper -q

import os
import json
import glob
from faster_whisper import WhisperModel
from tqdm import tqdm

CACHE_FILE = "transcripts_cache.json"

def load_cache() -> dict:
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_cache(cache: dict):
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f)

def resolve_audio_path(folder: str, filename: str) -> str:
    direct = os.path.join(folder, filename)
    if os.path.exists(direct):
        return direct
    for ext in [".wav", ".mp3", ".flac", ".ogg", ".m4a"]:
        candidate = os.path.join(folder, filename + ext)
        if os.path.exists(candidate):
            return candidate
    matches = glob.glob(os.path.join(folder, filename + ".*"))
    if matches:
        return matches[0]
    return direct

def transcribe(folder: str, filename: str) -> str:
    audio_path = resolve_audio_path(folder, filename)
    if not os.path.exists(audio_path):
        print(f"  [WARN] File not found: {audio_path}")
        return ""
    try:
        segments, _ = whisper_model.transcribe(audio_path, beam_size=1, language="en")
        return " ".join(seg.text for seg in segments).strip()
    except Exception as e:
        print(f"  [WARN] Transcription failed for {audio_path}: {e}")
        return ""

def transcribe_with_cache(folder: str, filename: str, cache: dict) -> str:
    if filename in cache:
        return cache[filename]
    result = transcribe(folder, filename)
    cache[filename] = result
    return result

whisper_model = WhisperModel("base", device=device, compute_type="int8")
cache = load_cache()

print("Transcribing training set …")
for fn in tqdm(train_df["filename"]):
    transcribe_with_cache(TRAIN_AUDIO, fn, cache)
save_cache(cache)
train_df["transcript"] = train_df["filename"].map(cache).fillna("")

print("Transcribing test set …")
for fn in tqdm(test_df["filename"]):
    transcribe_with_cache(TEST_AUDIO, fn, cache)
save_cache(cache)
test_df["transcript"] = test_df["filename"].map(cache).fillna("")

print("Done!")
print(f"Train transcripts: {(train_df['transcript'] != '').sum()}/{len(train_df)} successful")
print(f"Test  transcripts: {(test_df['transcript']  != '').sum()}/{len(test_df)}  successful")

Transcribing training set …


100%|██████████| 409/409 [40:54<00:00,  6.00s/it] 


Transcribing test set …


100%|██████████| 197/197 [03:22<00:00,  1.03s/it]

✅ Done!
Train transcripts: 409/409 successful
Test  transcripts: 197/197  successful


In [22]:
BERT_MODEL = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()

def get_bert_embedding(text: str) -> np.ndarray:
    """Mean-pool the last hidden state → 768-d vector."""
    if not text:
        return np.zeros(768)
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
        
    emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return emb

print("Encoding training transcripts …")
train_bert = np.vstack([get_bert_embedding(t) for t in tqdm(train_df["transcript"])])

print("Encoding test transcripts …")
test_bert  = np.vstack([get_bert_embedding(t) for t in tqdm(test_df["transcript"])])

del bert_model
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding training transcripts …


100%|██████████| 409/409 [01:38<00:00,  4.14it/s]


Encoding test transcripts …


100%|██████████| 197/197 [00:46<00:00,  4.19it/s]


In [23]:
def extract_audio_features(audio_path: str) -> np.ndarray:
    """
    Extract:
      - MFCCs (40 coefficients): mean + std → 80
      - Chroma:  mean + std → 24
      - Spectral contrast: mean + std → 14
      - ZCR: mean + std → 2
      - RMS energy: mean + std → 2
      Total: 122 features
    """
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception:
        return np.zeros(122)

    features = []

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    features += [mfcc.mean(axis=1), mfcc.std(axis=1)]

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features += [chroma.mean(axis=1), chroma.std(axis=1)]

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    features += [contrast.mean(axis=1), contrast.std(axis=1)]

    zcr = librosa.feature.zero_crossing_rate(y)
    features += [zcr.mean(axis=1), zcr.std(axis=1)]

    rms = librosa.feature.rms(y=y)
    features += [rms.mean(axis=1), rms.std(axis=1)]

    return np.concatenate(features)

print("Extracting audio features (train) …")
train_audio_feats = np.vstack([
    extract_audio_features(os.path.join(TRAIN_AUDIO, fn))
    for fn in tqdm(train_df["filename"])
])

print("Extracting audio features (test) …")
test_audio_feats  = np.vstack([
    extract_audio_features(os.path.join(TEST_AUDIO, fn))
    for fn in tqdm(test_df["filename"])
])

Extracting audio features (train) …


100%|██████████| 409/409 [00:03<00:00, 125.20it/s]


Extracting audio features (test) …


100%|██████████| 197/197 [00:00<00:00, 15463.52it/s]


In [25]:
import re

def linguistic_features(text: str) -> np.ndarray:
    """
    Hand-crafted features:
      - Word count, sentence count, avg words/sentence
      - Type-token ratio (vocabulary richness)
      - Avg word length
      - Filler-word rate ("um", "uh", "like", "you know")
      - Punctuation density
    """
    if not text:
        return np.zeros(7)

    words      = text.split()
    sentences  = re.split(r'[.!?]+', text)
    sentences  = [s for s in sentences if s.strip()]
    num_words  = len(words)
    num_sents  = max(len(sentences), 1)
    vocab      = set(w.lower() for w in words)
    fillers    = {"um", "uh", "like", "hmm", "ah", "er", "you know"}
    filler_cnt = sum(1 for w in words if w.lower() in fillers)

    return np.array([
        num_words,
        num_sents,
        num_words / num_sents,
        len(vocab) / max(num_words, 1),          
        np.mean([len(w) for w in words]) if words else 0,
        filler_cnt / max(num_words, 1),
        len(re.findall(r'[,;:—]', text)) / max(num_words, 1)
    ])

print("Computing linguistic features …")
train_ling = np.vstack([linguistic_features(t) for t in train_df["transcript"]])
test_ling  = np.vstack([linguistic_features(t) for t in test_df["transcript"]])

Computing linguistic features …


In [26]:
X_train = np.hstack([train_bert, train_audio_feats, train_ling])
X_test  = np.hstack([test_bert,  test_audio_feats,  test_ling])
y_train = train_df["label"].values   # adjust column name if different

print(f"Feature matrix shape — train: {X_train.shape}, test: {X_test.shape}")

Feature matrix shape — train: (409, 897), test: (197, 897)


In [29]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [30]:
kf      = KFold(n_splits=5, shuffle=True, random_state=42)
oof     = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
pearson_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = Ridge(alpha=1.0)
    model.fit(X_tr, y_tr)

    val_pred  = model.predict(X_val)
    oof[val_idx] = val_pred
    test_preds  += model.predict(X_test) / kf.n_splits

    r, _ = pearsonr(y_val, val_pred)
    pearson_scores.append(r)
    print(f"  Fold {fold+1} — Pearson r = {r:.4f}")

overall_r, _ = pearsonr(y_train, oof)
print(f"\nOOF Pearson r = {overall_r:.4f}")
print(f"Mean CV Pearson r = {np.mean(pearson_scores):.4f} ± {np.std(pearson_scores):.4f}")

  Fold 1 — Pearson r = 0.3830
  Fold 2 — Pearson r = 0.3358
  Fold 3 — Pearson r = 0.6306
  Fold 4 — Pearson r = 0.4404
  Fold 5 — Pearson r = 0.5334

OOF Pearson r = 0.4731
Mean CV Pearson r = 0.4646 ± 0.1059


In [34]:
test_preds = np.clip(test_preds, 0, 5)

In [35]:
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label":    test_preds
})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved")
print(submission.head(10))


submission.csv saved
      filename     label
0    audio_141  2.340146
1    audio_114  2.285614
2     audio_17  2.958174
3     audio_76  3.415228
4    audio_156  2.394961
5   audio_13_1  2.983673
6     audio_70  2.409459
7     audio_56  4.963419
8     audio_19  4.546220
9  audio_158_1  3.055270
